# Barkour/MJX/Brax Stage-M Pair-Gated Speed 0.85 BOUND Resume on Colab

This notebook runs the repository's `src/training.py` entrypoint on a Colab GPU runtime for the explicit `bound_stage_m_pair_gated_speed_085` experiment, resumed from the Stage-K final checkpoint. It keeps the familiar Drive-backed upload/checkpoint workflow from the previous Colab runs.

Use **Runtime > Change runtime type > GPU** before running the notebook. The target command remains `python src/training.py` with explicit `--experiment bound_stage_m_pair_gated_speed_085` and `--resume_checkpoint`; import paths, artifact paths, and noninteractive plotting/log capture are configured around it.

Persistence model: Colab still discards the VM, installed packages, and `/content` when the runtime terminates. This notebook mounts Google Drive and writes checkpoints, logs, metrics, environment metadata, uploaded source zips, and optional restored checkpoints under `MyDrive/STL-based-Quadruped-Locomotion/` so training artifacts survive runtime resets.


In [ ]:
# Colab hardware check
!nvidia-smi
import platform, sys
print('python:', sys.version)
print('platform:', platform.platform())


In [ ]:
# Mount Google Drive and define persistent project paths.
from google.colab import drive
from pathlib import Path

DRIVE_MOUNT = Path('/content/drive')
drive.mount(str(DRIVE_MOUNT))

DRIVE_PROJECT_DIR = DRIVE_MOUNT / 'MyDrive' / 'STL-based-Quadruped-Locomotion'
DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / 'runs'
DRIVE_SOURCES_DIR = DRIVE_PROJECT_DIR / 'source_zips'
DRIVE_CHECKPOINTS_DIR = DRIVE_PROJECT_DIR / 'checkpoints'
for path in (DRIVE_PROJECT_DIR, DRIVE_RUNS_DIR, DRIVE_SOURCES_DIR, DRIVE_CHECKPOINTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

Path('/content/.drive_project_dir').write_text(str(DRIVE_PROJECT_DIR))
Path('/content/.drive_runs_dir').write_text(str(DRIVE_RUNS_DIR))
Path('/content/.drive_sources_dir').write_text(str(DRIVE_SOURCES_DIR))
Path('/content/.drive_checkpoints_dir').write_text(str(DRIVE_CHECKPOINTS_DIR))

print('Persistent project dir:', DRIVE_PROJECT_DIR)
print('Persistent runs dir:', DRIVE_RUNS_DIR)
print('Persistent checkpoints dir:', DRIVE_CHECKPOINTS_DIR)


In [ ]:
%%bash
set -e
# Install the repo's portable dependencies. JAX CUDA follows the official pip-managed CUDA path.
# Runtime restart is usually not required unless Colab had a conflicting preinstalled JAX stack loaded already.
python -m pip install -q --upgrade pip setuptools wheel
python -m pip install -q \
  'jax[cuda12]==0.7.1' \
  absl-py==2.3.1 brax==0.13.0 chex==0.1.91 etils==1.13.0 flax==0.11.2 \
  jaxopt==0.8.5 matplotlib==3.10.6 mediapy==1.2.4 ml-collections==1.1.0 \
  mujoco==3.3.5 mujoco-mjx==3.3.5 numpy==2.3.3 optax==0.2.5 \
  orbax-checkpoint==0.11.24 pandas==2.3.3 scipy==1.16.1 tensorboardx==2.6.4


In [ ]:
# Upload a zip of the GitHub repository, persist it to Drive, then fetch MuJoCo Menagerie.
from google.colab import files
from pathlib import Path
import shutil
import zipfile

DRIVE_PROJECT_DIR = Path(Path('/content/.drive_project_dir').read_text().strip())
DRIVE_SOURCES_DIR = Path(Path('/content/.drive_sources_dir').read_text().strip())

%cd /content
!rm -rf STL-based-Quadruped-Locomotion mujoco_menagerie _repo_upload

uploaded = files.upload()
zip_paths = [Path(name) for name in uploaded if name.endswith('.zip')]
if not zip_paths:
    raise ValueError('Upload a .zip file of the STL-based-Quadruped-Locomotion repository.')

source_zip = zip_paths[0]
drive_zip = DRIVE_SOURCES_DIR / source_zip.name
shutil.copy2(source_zip, drive_zip)
print('Saved uploaded source zip to Drive:', drive_zip)

extract_root = Path('/content/_repo_upload')
with zipfile.ZipFile(source_zip) as zf:
    zf.extractall(extract_root)

repo_candidates = [p for p in extract_root.rglob('src/training.py') if '__MACOSX' not in p.parts]
if not repo_candidates:
    raise FileNotFoundError('Could not find src/training.py inside the uploaded zip.')

repo_src = repo_candidates[0].parents[1]
repo_dst = Path('/content/STL-based-Quadruped-Locomotion')
shutil.copytree(repo_src, repo_dst)
%cd /content/STL-based-Quadruped-Locomotion
uploaded_commit = input('Optional: paste the source git commit hash for this zip, then press Enter: ').strip()
Path('.uploaded_git_commit.txt').write_text(uploaded_commit or 'not available: uploaded zip did not include .git metadata')
Path('.drive_source_zip.txt').write_text(str(drive_zip))

!git clone --depth 1 https://github.com/google-deepmind/mujoco_menagerie.git
!git -C mujoco_menagerie rev-parse HEAD


In [ ]:
# Upload/persist the baseline checkpoint zip to Drive for a resume run.
# For a checkpoint folder, zip the whole checkpoint directory first, then upload that zip here.
# The zip may contain the checkpoint directly or a parent path such as models8/ckpts/date/step.
from google.colab import files
from pathlib import Path
import shutil
import zipfile

UPLOAD_RESUME_CHECKPOINT = True
USE_EXISTING_DRIVE_CHECKPOINT = False
RESUME_CHECKPOINT_LABEL = 'stage_k_50872320'
DRIVE_CHECKPOINTS_DIR = Path(Path('/content/.drive_checkpoints_dir').read_text().strip())
PERSISTED_CHECKPOINT_PATH = DRIVE_CHECKPOINTS_DIR / RESUME_CHECKPOINT_LABEL

def looks_like_orbax_checkpoint(path):
    return (path / '_CHECKPOINT_METADATA').exists() or ((path / '_METADATA').exists() and (path / 'manifest.ocdbt').exists())

def find_checkpoint_path(root):
    if looks_like_orbax_checkpoint(root):
        return root
    candidates = [p for p in root.rglob('*') if p.is_dir() and looks_like_orbax_checkpoint(p)]
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        print('Found multiple checkpoint-like directories:')
        for p in candidates:
            print('  ', p)
        raise ValueError('Multiple checkpoint directories found; upload a zip containing only one checkpoint.')
    children = [p for p in root.iterdir() if p.name != '__MACOSX']
    if len(children) == 1 and children[0].is_dir():
        return find_checkpoint_path(children[0])
    raise FileNotFoundError('Could not find an Orbax checkpoint directory in the uploaded zip.')

if UPLOAD_RESUME_CHECKPOINT:
    uploaded_ckpt = files.upload()
    ckpt_zips = [Path(name) for name in uploaded_ckpt if name.endswith('.zip')]
    if not ckpt_zips:
        raise ValueError('Upload a .zip containing the checkpoint directory contents.')
    if PERSISTED_CHECKPOINT_PATH.exists():
        shutil.rmtree(PERSISTED_CHECKPOINT_PATH)
    PERSISTED_CHECKPOINT_PATH.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ckpt_zips[0]) as zf:
        zf.extractall(PERSISTED_CHECKPOINT_PATH)
    actual = find_checkpoint_path(PERSISTED_CHECKPOINT_PATH)
    Path('/content/.resume_checkpoint_path').write_text(str(actual))
    print('Persisted resume checkpoint to:', actual)
elif USE_EXISTING_DRIVE_CHECKPOINT:
    actual = find_checkpoint_path(PERSISTED_CHECKPOINT_PATH)
    Path('/content/.resume_checkpoint_path').write_text(str(actual))
    print('Using existing Drive checkpoint:', actual)
else:
    Path('/content/.resume_checkpoint_path').unlink(missing_ok=True)
    print('No resume checkpoint selected. This notebook is configured below to require one for v1 resume.')
    print('Expected persisted checkpoint path if already uploaded:', PERSISTED_CHECKPOINT_PATH)


In [ ]:
# Compatibility patch only: redirect artifacts with BARKOUR_OUTPUT_ROOT and BARKOUR_METRICS_PATH.
# This does not alter training parameters or reward/environment behavior.
from pathlib import Path

training = Path('src/training.py')
text = training.read_text()
old = "today = date.today().strftime('%Y_%m_%d')\npath_ = f'/home/matasever/projects/Quadrupeds_STLReward/models8/ckpts/{today}'"
new = "today = date.today().strftime('%Y_%m_%d')\noutput_root = os.environ.get(\n    'BARKOUR_OUTPUT_ROOT',\n    '/home/matasever/projects/Quadrupeds_STLReward/models8',\n)\npath_ = f'{output_root}/ckpts/{today}'"
if old in text:
    text = text.replace(old, new)
text = text.replace("model_path = f'/home/matasever/projects/Quadrupeds_STLReward/models8/{today}'", "model_path = f'{output_root}/{today}'")
text = text.replace(
    "df_metrics.to_csv('metricssaved.csv', sep='|', index=False)",
    "df_metrics.to_csv(os.environ.get('BARKOUR_METRICS_PATH', 'metricssaved.csv'), sep='|', index=False)",
)
training.write_text(text)

# Compatibility patch only: keep reset/step reward metric pytrees identical for JAX scan.
# Missing configured metric keys are logging placeholders set to 0.0; scalar reward remains r * dt.
barkour = Path('src/Barkour.py')
text = barkour.read_text()
needle = (
    '         # "support_dist": support_dist,\n'
    '          }\n'
    '      \n'
    '      \n'
    '      reward = jp.clip(r * self.dt, -100.0, 1000.0)'
)
replacement = (
    '         # "support_dist": support_dist,\n'
    '          }\n'
    '      rewards = {\n'
    '          k: rewards.get(k, 0.0) for k in self.reward_config.rewards.scales.keys()\n'
    '      }\n'
    '      \n'
    '      \n'
    '      reward = jp.clip(r * self.dt, -100.0, 1000.0)'
)
if needle in text and 'rewards.get(k, 0.0)' not in text:
    text = text.replace(needle, replacement)
barkour.write_text(text)

if Path('.git').exists():
    import subprocess
    subprocess.run(['git', 'diff', '--', 'src/training.py', 'src/Barkour.py'], check=False)
else:
    print('Uploaded zip has no .git directory; compatibility patches were applied in-place.')


In [ ]:
# Create the persistent Stage-M resume output directory in Google Drive and capture metadata.
from datetime import datetime
from pathlib import Path
import os, shlex, subprocess, textwrap

EXPERIMENT_NAME = 'bound_stage_m_pair_gated_speed_085'
RUN_NAME = 'bound_stage_m_pair_gated_speed_085_resume'
REQUIRE_RESUME_CHECKPOINT = True

resume_checkpoint = ''
resume_path_file = Path('/content/.resume_checkpoint_path')
if resume_path_file.exists():
    resume_checkpoint = resume_path_file.read_text().strip()
if REQUIRE_RESUME_CHECKPOINT and not resume_checkpoint:
    raise FileNotFoundError('No resume checkpoint is configured. Run the checkpoint upload cell first.')

DRIVE_RUNS_DIR = Path(Path('/content/.drive_runs_dir').read_text().strip())
RUN_DIR = DRIVE_RUNS_DIR / f"{RUN_NAME}_colab_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
Path('.colab_run_dir').write_text(str(RUN_DIR))
Path('.colab_experiment_name').write_text(EXPERIMENT_NAME)
Path('.colab_run_name').write_text(RUN_NAME)

env_info = subprocess.check_output([
    'python', '-c',
    "import platform, jax; print('platform:', platform.platform()); print('jax:', jax.__version__); print('devices:', jax.devices())"
], text=True)
(RUN_DIR / 'environment_info.txt').write_text(env_info)

def safe_output(cmd, fallback):
    try:
        return subprocess.check_output(cmd, text=True, stderr=subprocess.DEVNULL)
    except Exception:
        return fallback + '\n'

uploaded_commit = Path('.uploaded_git_commit.txt').read_text().strip() if Path('.uploaded_git_commit.txt').exists() else 'not available: uploaded zip did not include .git metadata'
(RUN_DIR / 'git_commit.txt').write_text(safe_output(['git', 'rev-parse', 'HEAD'], uploaded_commit))
(RUN_DIR / 'git_branch.txt').write_text(safe_output(['git', 'branch', '--show-current'], 'not available: uploaded zip did not include .git metadata'))
(RUN_DIR / 'mujoco_menagerie_commit.txt').write_text(subprocess.check_output(['git', '-C', 'mujoco_menagerie', 'rev-parse', 'HEAD'], text=True))
if Path('.drive_source_zip.txt').exists():
    (RUN_DIR / 'drive_source_zip.txt').write_text(Path('.drive_source_zip.txt').read_text())
if resume_checkpoint:
    (RUN_DIR / 'resume_checkpoint.txt').write_text(resume_checkpoint + '\n')
subprocess.run('python -m pip freeze > ' + str(RUN_DIR / 'pip_freeze.txt'), shell=True, check=True)

cmd_args = [
    'python', 'src/training.py',
    '--experiment', EXPERIMENT_NAME,
    '--run_name', RUN_NAME,
    '--output_root', str(RUN_DIR / 'models8'),
]
if resume_checkpoint:
    cmd_args.extend(['--resume_checkpoint', resume_checkpoint])
cmd = (
    f"PYTHONPATH=src:configs BARKOUR_OUTPUT_ROOT={RUN_DIR / 'models8'} "
    f"BARKOUR_METRICS_PATH={RUN_DIR / 'metricssaved.csv'} MPLBACKEND=Agg PYTHONUNBUFFERED=1 "
    + ' '.join(shlex.quote(x) for x in cmd_args)
)
(RUN_DIR / 'training_command.txt').write_text(cmd + '\n')
print(env_info)
print('Persistent RUN_DIR:', RUN_DIR)
print('experiment:', EXPERIMENT_NAME)
print('run name:', RUN_NAME)
print('resume checkpoint:', resume_checkpoint or 'none')
print('command:', cmd)


In [ ]:
%%bash
set -o pipefail
# Minimal preflight: import and instantiate the configured experiment environment.
PYTHONPATH=src:configs python - <<'PY' 2>&1 | tee "$(cat .colab_run_dir)/preflight_env.log"
from Barkour import BarkourEnv
from pathlib import Path
experiment = Path('.colab_experiment_name').read_text().strip()
env = BarkourEnv(experiment=experiment)
print('env ok')
print('experiment', experiment)
print('dt', env.dt)
print('action_size', env.action_size)
print('observation_size', env.observation_size)
PY


In [ ]:
%%bash
set -o pipefail
# Real Stage-M resume training run. PPO hyperparameters remain the src/training.py defaults except the shorter first-run horizon below.
RUN_DIR=$(cat .colab_run_dir)
EXPERIMENT_NAME=$(cat .colab_experiment_name)
RUN_NAME=$(cat .colab_run_name)
mkdir -p "$RUN_DIR" "$RUN_DIR/models8"
export PYTHONPATH=src:configs
export BARKOUR_OUTPUT_ROOT="$RUN_DIR/models8"
export BARKOUR_METRICS_PATH="$RUN_DIR/metricssaved.csv"
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
ARGS=(--experiment "$EXPERIMENT_NAME" --run_name "$RUN_NAME" --output_root "$RUN_DIR/models8" --num_timesteps 50000000 --num_evals 10)
if [[ -s /content/.resume_checkpoint_path ]]; then
  ARGS+=(--resume_checkpoint "$(cat /content/.resume_checkpoint_path)")
else
  echo 'Missing /content/.resume_checkpoint_path; refusing to start a scratch run.' >&2
  exit 1
fi
python src/training.py "${ARGS[@]}" 2>&1 | tee "$RUN_DIR/train.log"


In [ ]:
# Generate reward-vs-environment-step plot from the Drive-backed metricssaved.csv.
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt

RUN_DIR = Path(Path('.colab_run_dir').read_text().strip())
csv_src = RUN_DIR / 'metricssaved.csv'
if not csv_src.exists():
    local_fallback = Path('metricssaved.csv')
    if local_fallback.exists():
        shutil.copy2(local_fallback, csv_src)
    else:
        raise FileNotFoundError('metricssaved.csv not found in Drive or local cwd; training may not have reached the first CSV write at ~20M env steps.')

df = pd.read_csv(csv_src, sep='|')
last_reward = float(df['eval_episode_reward'].dropna().iloc[-1])
last_steps = int(df['num_steps'].dropna().iloc[-1])
plt.figure(figsize=(8, 5))
plt.plot(df['num_steps'], df['eval_episode_reward'], marker='o')
plt.xlabel('# environment steps')
plt.ylabel('reward per episode')
plt.title(f'y={last_reward:.3f}')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plot_path = RUN_DIR / 'reward_curve.png'
plt.savefig(plot_path, dpi=180)

env_text = (RUN_DIR / 'environment_info.txt').read_text().strip()
summary = (
    f"# Default Barkour/MJX/Brax Reproduction Summary\n\n"
    f"- Exact command: `{(RUN_DIR / 'training_command.txt').read_text().strip()}`\n"
    f"- Training parameter changes: none\n"
    f"- Compatibility changes: output root env var; Drive-backed metrics path; missing reward metric keys padded for JAX pytree consistency\n"
    f"- Git commit: `{(RUN_DIR / 'git_commit.txt').read_text().strip()}`\n"
    f"- Total environment steps reached: `{last_steps}`\n"
    f"- Final reward value: `{last_reward:.6f}`\n"
    f"- Plot: `{plot_path}`\n\n"
    f"## Environment\n\n```\n{env_text}\n```\n"
)
(RUN_DIR / 'reproduction_summary.md').write_text(summary)
print(summary)
print('saved:', plot_path)


In [ ]:
# Package artifacts into the same Drive-backed run directory.
from pathlib import Path
import shutil
RUN_DIR = Path(Path('.colab_run_dir').read_text().strip())
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('artifact zip saved in Drive:', archive)
print('run directory remains available at:', RUN_DIR)
